# 00 · Medallón del COES: demanda por área del SEIN

Baja el anexo diario del Informe de Evaluación de la Operación Diaria (IEOD) del COES, de 2015 a hoy, y arma la
demanda cada 30 minutos del SEIN y de sus tres áreas (Norte, Centro, Sur). Todo queda en el Drive, dentro de
`coes/`, junto a las demás capas del proyecto.

| Paso | Qué hace |
|---|---|
| Punto de control | Un anexo por año (el 2 de marzo), leído con el mismo lector: Norte + Centro + Sur = SEIN |
| Bronze | Los anexos de cada día, sin tocar, con su procedencia (ruta en el COES, sha256, hojas, fecha de descarga). Se puede cortar y retomar |
| Silver | La demanda por área cada 30 minutos y su auditoría |

Son unos 4,300 días y ~400 MB por año: corre en Colab, no en local.

## Instalación

In [ ]:
import os
import sys

RAMA = "feature/ingesta-coes"  # cambiar a "develop" cuando se fusione
!pip install -q "git+https://github.com/R0SEWT/tetouan-power-forecast@{RAMA}"

In [ ]:
import datetime as dt
import shutil

from IPython.display import Markdown, display

from tetouan_power_forecast import coes
from tetouan_power_forecast.medallon import usar_drive

# Con EN_DRIVE = True las capas se escriben directo en el Drive (pide permiso en una ventana emergente).
# Con False se escriben en el disco de la máquina de Colab y la última celda las copia al Drive.
EN_DRIVE = False
if EN_DRIVE:
    raiz = usar_drive()  # Mi unidad/datalake/tetouan-power-forecast
else:
    os.environ["TETOUAN_DATA"] = "/content/datos"

## Punto de control: un anexo por año

Antes de la descarga masiva se lee el 2 de marzo de cada año con el mismo lector. Si algún año no trae
`DEMANDA_AREAS` o la suma de las áreas no da el SEIN, se revisa el lector antes de seguir.

In [ ]:
control = coes.un_anexo_por_anio()
display(control)
assert (control["estado"] == "ok").all() and (control["desvio_max_mw"] <= 1).all(), "revisar el lector antes de seguir"

## Bronze: los anexos de cada día

Retoma donde quedó: los días ya registrados en la procedencia no se vuelven a pedir. Los días sin anexo quedan
marcados para revisarlos. Va por año en orden de prioridad: la prueba, la validación y los años con El Niño primero.

In [ ]:
# Primero la prueba (2026), la validación (2025) y los años con Niño; si la noche no alcanza, eso ya está
PRIORIDAD = [2026, 2025, 2024, 2023, 2017, 2022, 2021, 2020, 2019, 2018, 2016, 2015]
for anio in PRIORIDAD:
    hasta = None if anio == dt.date.today().year else dt.date(anio, 12, 31)
    coes.bronze_ieod(desde=dt.date(anio, 1, 1), hasta=hasta, hilos=4)

## Silver: demanda por área cada 30 minutos

In [ ]:
ruta = coes.silver_areas(forzar=True)
display(Markdown(ruta.with_name("auditoria.md").read_text()))

## Copiar al Drive

Solo si se corrió con `EN_DRIVE = False`: copia `coes/` del disco de Colab a la carpeta del proyecto en el
Drive. La copia va entre servidores de Google y retoma lo que falte.

In [ ]:
if not EN_DRIVE:
    destino = usar_drive() / "coes"
    shutil.copytree("/content/datos/coes", destino, dirs_exist_ok=True)
    print("copiado a", destino)